1. Dataset

IMDb Reviews

In [4]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PVNS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PVNS\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\PVNS\AppData\Roaming\nltk_data...


True

2. Import Libraries

In [5]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

3. Data Understanding

In [6]:
df = pd.read_csv("C:/Users/PVNS/Downloads/Dataset/archive/Review.csv")

print(df.head())
print(df.shape)
print(df['sentiment'].value_counts())

  sentiment                                             review
0  Negative  I had no background knowledge of this movie be...
1  Negative  I am a huge Jane Austen fan and I ordered the ...
2  Negative  Nothing to say but Wow! Has anyone actually ha...
3  Negative  i like Jane Austin novels. I love Pride and Pr...
4  Negative  In this day and age of incredible special movi...
(10000, 2)
sentiment
Negative    5081
Positive    4919
Name: count, dtype: int64


4. NLP Preprocessing

In [8]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # remove special chars
    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    
    return " ".join(words)

In [9]:
df['clean_text'] = df['review'].apply(preprocess)

5. Feature Engineering

In [ ]:
Bag of Words

In [10]:
bow = CountVectorizer()
X_bow = bow.fit_transform(df['clean_text'])

TF-IDF

In [11]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])

Train-Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, df['sentiment'], test_size=0.2, random_state=42
)

7. Train Models

Logistic Regression

In [13]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

Naive Bayes

In [14]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

Decision Tree

In [15]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

8. Evaluation

In [16]:
def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    }

In [17]:
print("Logistic Regression:", evaluate(y_test, y_pred_lr))
print("Naive Bayes:", evaluate(y_test, y_pred_nb))
print("Decision Tree:", evaluate(y_test, y_pred_dt))

Logistic Regression: {'Accuracy': 0.863, 'Precision': 0.8637729530360461, 'Recall': 0.863, 'F1 Score': 0.8630057540230162}
Naive Bayes: {'Accuracy': 0.834, 'Precision': 0.8372681845485841, 'Recall': 0.834, 'F1 Score': 0.8333489377749843}
Decision Tree: {'Accuracy': 0.6905, 'Precision': 0.6904803457137025, 'Recall': 0.6905, 'F1 Score': 0.6904886190435809}


In [21]:
lr_result = evaluate(y_test, y_pred_lr)
nb_result = evaluate(y_test, y_pred_nb)
dt_result = evaluate(y_test, y_pred_dt)

9. Comparison Table

In [22]:
results = pd.DataFrame([
    {"Model": "Logistic Regression", **lr_result},
    {"Model": "Naive Bayes", **nb_result},
    {"Model": "Decision Tree", **dt_result}
])

print(results)

                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression    0.8630   0.863773  0.8630  0.863006
1          Naive Bayes    0.8340   0.837268  0.8340  0.833349
2        Decision Tree    0.6905   0.690480  0.6905  0.690489


####Insights & Observations
###Preprocessing Impact

#Text preprocessing played a crucial role in improving model performance. Steps like lowercasing, removing punctuation, stopword removal, and stemming helped in reducing noise and dimensionality of the data. This resulted in better feature quality and improved model accuracy.

###TF-IDF vs Bag of Words

#TF-IDF performed better compared to Bag of Words. This is because TF-IDF reduces the importance of very common words and gives more weight to informative words that are unique to a document. In contrast, Bag of Words only considers frequency, which may give high importance to irrelevant common terms.

###Best Performing Model

#Logistic Regression achieved the best performance among all models in terms of Accuracy and F1 Score. It handled the high-dimensional sparse data effectively and was able to generalize well on unseen data.

###Model Comparison
#Logistic Regression: Highest accuracy and balanced performance across all metrics
#Naive Bayes: Fast and efficient, but slightly lower accuracy due to its strong independence assumption
#Decision Tree: Lower performance compared to others, prone to overfitting on text data

###Final Conclusion

#Overall, the combination of proper preprocessing, TF-IDF feature extraction, and Logistic Regression model gave the best results for sentiment analysis. This pipeline is effective for handling textual data and can be further improved using advanced techniques like Word2Vec or ensemble models